You will conduct experiments with the aim of determining the velocity field of a flow using PIV. The experiments are carried out in a recirculating-water glass tank measuring 30 x 35 x 200 cm³. Its modest volume is well-suited for the space constraints of a practical laboratory exercise. However, you will notice that it has **consequences on the established flow** that you will analyze. The pump can produce an adjustable flow rate of up to approximately 200 m$^3$/h, thus creating a controlled recirculating flow in the aquarium.

## Experimental setup

### Seeding particles

The seeding particles used are polyamide particles (PSP-50, ref. 9080A5011). These particles are produced by polymerization. They are round, but not perfectly spherical. The size distribution (diameter) of each particle is between 30 and 70 μm, with an average of 50 μm. Their density is 1.03 g/cm$^3$, very close to that of water, which therefore limits their sedimentation. Their refractive index is 1.5.

### Light source

Illumination is provided by the Fiber-Lite® Mi-LED light generator (Dolan-Jenner). This system delivers white light with a color temperature of 5000 K and represents a modern and economical alternative to conventional 150 W halogen sources.

### Image recording

The images are acquired by the FlowSense 2M-165 camera, which connects directly to the computer via a USB 3 port, thus integrating the system directly within the computer without requiring an additional acquisition card. This camera offers a resolution of 1920 × 1200 pixels (2.3 megapixels), with a maximum acquisition frequency of 160 frames per second. It is therefore well-suited for time-resolved PIV for slow to moderate flows. Its pixel size is 5.9 μm, and the quantum efficiency of its sensor is greater than 70%, particularly adapted to the wavelengths of green light from lasers or LEDs.
It uses a CMOS (Complementary Metal-Oxide-Semiconductor) sensor, which is widely used in modern PIV systems for its high acquisition rate.

## Acquisition protocol

This is a **quick-reference checklist** meant to be read once before your rotation slot, not
a tutorial. The station is shared: one binome at a time. **Bring a USB key**.

### Before your slot: target realizations per series

- The mean velocity field estimated from `N` independent realizations converges as roughly $\sigma / \sqrt{N}$: doubling `N` only buys you a $\sqrt 2$ reduction in noise. Default: **~150 independent pairs** per flow/regime for the batch statistics — going from 150 to 500 only buys a factor ~1.8 in precision for over 3x the capture/transfer time.

### One-time setup

- **Calibration image.** With the empty aquarium (no obstacle) and the
   illumination sheet in place, image the dot target held flat in the
   sheet plane. Capture **one single frame**. Note
   which two graduations you can read clearly and their spacing in mm.

### Per case (flow × regime)

1. Mount the obstacle (cylinder / step insert) or remove it (free
   stream).
2. Set the pump regime (low/mid/high). Wait for the flow to visibly
   settle before capturing.
3. **Choose `dt`** (interframe time): a
   particle should move roughly a **quarter of the final interrogation window** between the two frames. Too
   large a `dt` loses particles out-of-plane; too small gives displacements too close to the pixel-locking regime discussed in the Lecture. **Write down the dt used** (1 / frame rate, in seconds), there is no way to recover it from the images
   themselves afterwards. 
4. Capture your target number of images.
5. Copy the raw TIFF frames for this case onto a folder on your USB key (one folder per flow/regime — you'll sort and convert them, back at your laptop — see the Appendix at the end of this notebook for the conversion).

## Summary

During this practical session, you will get familiarized with a simple experimental setup designed to record particle images of a flow for which the velocity field is to be estimated. 

> **Assignement.** You will be asked to study three particular flows:
>
>  1. a free flow (without any obstacle)
>  2. the flow that develops around/behind a cylinder
>  3. the flow that develops behind a step
>
>each of which with three flow regimes, i.e. for three different flow rates as given by the pump. Record sufficiently substantial samples to allow you to subsequently conduct statistical studies.
>
> An in-depth analysis of the obtained images will be expected. Among the points to be discussed, without being exhaustive, should include: the average size of the particle images, their density, and the dynamic range of the gray levels. To do this, you could for example use the `regionprops` library adapted from Matlab:

In [ ]:
# Import necessary modules
import numpy as np
import matplotlib.pyplot as plt
#matplotlib inline
from openpiv import tools
from skimage.measure import label, regionprops

# Load a sample image
image = tools.imread('your image.tif')
fig, ax = plt.subplots(figsize=(12, 10))
ax.imshow(image, cmap=plt.cm.gray)

# Use a threshold to segment bright particles
threshold = np.percentile(image, 90)  # adjust percentile as needed
binary_img = image > threshold

label_img = label(binary_img, background=0) # adjust background as needed
regions = regionprops(label_img, intensity_image=image)

for props in regions:
    # Use weighted centroid for better accuracy
    y0, x0 = props.weighted_centroid
    ax.plot(x0, y0, marker='x', color='r', markersize=6)

plt.show()

> You will then work on the collected databases by implementing a method for evaluating velocity fields using OpenPIV. You will detail the method used, emphasizing the choice of parameters for the PIV analysis.
>
> Finally, you will develop an analysis of the obtained velocity fields based on, among other things, considerations discussed in class with Mr. Druault.

## Appendix: converting your images

Use the function below to convert the raw TIFF frames of a folder into PNG. Non-8-bit TIFFs are rescaled to 8-bit first —
this is also what makes the PNGs much smaller than the raw TIFFs.

Call `convert_to_png(src_dir, dst_dir)` once per folder you captured. Pass `prefix="..."` if you also want to rename the files: the trailing number of each file name is kept and everything before it is replaced by your prefix.

In [ ]:
import pathlib
import re
import numpy as np
import imageio.v3 as iio

def convert_to_png(src_dir, dst_dir=None, prefix=None):
    """Convert every .tif/.tiff image of `src_dir` to a .png in `dst_dir`
    (default: next to the originals).

    prefix: if given, rename the files -- the trailing number of each name
    is kept (same zero-padding), everything before it is replaced by `prefix`.
    """
    src_dir = pathlib.Path(src_dir)
    dst_dir = pathlib.Path(dst_dir) if dst_dir else src_dir
    files = sorted(f for f in src_dir.iterdir() if f.suffix.lower() in ('.tif', '.tiff'))
    if not files:
        print(f"No .tif/.tiff files found in {src_dir}")
        return
    dst_dir.mkdir(parents=True, exist_ok=True)

    for f in files:
        img = iio.imread(f)
        if img.dtype != np.uint8:       # rescale to 8 bit -> much smaller PNG
            img = img.astype(np.float64)
            img -= img.min()
            if img.max() > 0:
                img = img / img.max() * 255.0
            img = img.astype(np.uint8)

        stem = f.stem
        if prefix is not None:
            m = re.match(r'^(.*?)(\d+)$', stem)
            stem = f"{prefix}{int(m.group(2)):0{len(m.group(2))}d}" if m else f"{prefix}{stem}"

        iio.imwrite(dst_dir / f"{stem}.png", img)
        print(f"{f.name} -> {stem}.png")
    print(f"Done: {len(files)} image(s) converted in {dst_dir}")


# Example -- adapt the paths to your USB key:
# convert_to_png("/Volumes/USB_KEY/cylinder_mid", "data/cylinder/mid")
# convert_to_png("/Volumes/USB_KEY/calib", "data/calibration")